# Approach 2 (PyTorch): 3D Landmarks + MobileFaceNet

Notebook ini pakai PyTorch dengan custom extractor berbasis konfigurasi:
- `dataset_root`
- `training_dataset`
- `test_dataset`

Input model:
- gambar wajah
- seluruh landmark MediaPipe yang tersedia di CSV (`x_i, y_i, z_i`)
- fusion image branch + landmark branch


In [ ]:
from __future__ import annotations

import csv
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "device:", DEVICE)

In [ ]:
@dataclass
class Config:
    image_size: int = 160
    batch_size: int = 32
    lr: float = 1e-3
    epochs: int = 10
    val_ratio: float = 0.2
    dataset_root: str = "output"
    training_dataset: str = "training_set"
    test_dataset: str = "testing_set"


CFG = Config()

print("dataset_root:", CFG.dataset_root)
print("training_dataset:", CFG.training_dataset)
print("test_dataset:", CFG.test_dataset)


In [ ]:
class OutputSplitExtractor:
    def __init__(self, split_root: Path, dataset_root: Path):
        self.split_root = split_root
        self.dataset_root = dataset_root

    def _class_dirs(self) -> List[Path]:
        if not self.split_root.exists():
            raise FileNotFoundError(f"Split folder tidak ada: {self.split_root}")
        return sorted([d for d in self.split_root.iterdir() if d.is_dir()])

    def _resolve_image_path(self, image_name: str, image_path_raw: str, class_dir: Path) -> Path:
        raw = Path(image_path_raw)
        if raw.is_absolute() and raw.exists():
            return raw

        candidates = [
            self.dataset_root.parent / raw,
            self.dataset_root / raw,
            Path.cwd().resolve() / raw,
            class_dir / image_name,
        ]
        for candidate in candidates:
            if candidate.exists():
                return candidate

        return class_dir / image_name

    def _extract_landmark_vector(self, row: Dict[str, str]) -> np.ndarray:
        # Ambil semua landmark x_i, y_i, z_i yang tersedia di CSV.
        values: List[float] = []
        i = 0
        while True:
            x = row.get(f"x_{i}")
            y = row.get(f"y_{i}")
            z = row.get(f"z_{i}")
            if x is None or y is None or z is None:
                break

            values.extend([float(x), float(y), float(z)])
            i += 1

        if not values:
            raise ValueError("Baris CSV tidak punya landmark x_i/y_i/z_i.")
        return np.asarray(values, dtype=np.float32)

    def extract(self) -> List[Dict[str, object]]:
        records: List[Dict[str, object]] = []
        expected_landmark_dim = None

        for class_dir in self._class_dirs():
            label_name = class_dir.name
            csv_path = class_dir / "landmarks.csv"
            if not csv_path.exists():
                continue

            with csv_path.open("r", encoding="utf-8", newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    image_name = row["image_name"]
                    image_path = self._resolve_image_path(
                        image_name,
                        row.get("image_path", ""),
                        class_dir,
                    )
                    if not image_path.exists():
                        continue

                    landmark_vec = self._extract_landmark_vector(row)
                    if expected_landmark_dim is None:
                        expected_landmark_dim = landmark_vec.shape[0]
                    elif landmark_vec.shape[0] != expected_landmark_dim:
                        # Skip row yang dimensi landmark-nya beda agar tensor shape konsisten.
                        continue

                    records.append(
                        {
                            "image_path": str(image_path),
                            "label_name": label_name,
                            "landmarks": landmark_vec,
                        }
                    )

        if not records:
            raise RuntimeError(f"Tidak ada record terbaca dari {self.split_root}")

        return records


In [ ]:
class FaceShapeHybridDataset(Dataset):
    def __init__(self, records: List[Dict[str, object]], label_to_idx: Dict[str, int], image_size: int = 160):
        self.records = records
        self.label_to_idx = label_to_idx
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        img = Image.open(rec["image_path"]).convert("RGB")
        img = self.transform(img)

        landmarks = torch.from_numpy(rec["landmarks"]).float()
        label = torch.tensor(self.label_to_idx[rec["label_name"]], dtype=torch.long)

        return {
            "image": img,
            "landmarks": landmarks,
            "label": label,
        }

In [ ]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int, s: int, groups: int = 1):
        super().__init__()
        p = (k - 1) // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU6(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DepthwiseSeparable(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int):
        super().__init__()
        self.dw = ConvBNReLU(in_ch, in_ch, k=3, s=stride, groups=in_ch)
        self.pw = ConvBNReLU(in_ch, out_ch, k=1, s=1)

    def forward(self, x):
        return self.pw(self.dw(x))


class MobileFaceNetBackbone(nn.Module):
    def __init__(self, embedding_dim: int = 128):
        super().__init__()
        self.features = nn.Sequential(
            ConvBNReLU(3, 64, k=3, s=2),
            DepthwiseSeparable(64, 64, stride=1),
            DepthwiseSeparable(64, 128, stride=2),
            DepthwiseSeparable(128, 128, stride=1),
            DepthwiseSeparable(128, 256, stride=2),
            DepthwiseSeparable(256, 256, stride=1),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, embedding_dim, bias=False)
        self.bn = nn.BatchNorm1d(embedding_dim)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.bn(self.fc(x))
        x = nn.functional.normalize(x, p=2, dim=1)
        return x


class HybridFaceShapeNet(nn.Module):
    def __init__(self, num_classes: int, landmark_dim: int, image_embedding_dim: int = 128):
        super().__init__()
        self.image_branch = MobileFaceNetBackbone(embedding_dim=image_embedding_dim)
        self.landmark_branch = nn.Sequential(
            nn.Linear(landmark_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
        )

        self.head = nn.Sequential(
            nn.Linear(image_embedding_dim + 64, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, image, landmarks):
        image_feat = self.image_branch(image)
        lm_feat = self.landmark_branch(landmarks)
        fused = torch.cat([image_feat, lm_feat], dim=1)
        logits = self.head(fused)
        return logits

In [ ]:
def resolve_dataset_root(cfg: Config) -> Path:
    root = Path(cfg.dataset_root).expanduser()
    if root.is_absolute():
        return root

    cwd = Path.cwd().resolve()
    candidates = [cwd / root, cwd.parent / root]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    return (cwd / root).resolve()


def build_loaders(cfg: Config):
    dataset_root = resolve_dataset_root(cfg)
    train_split_root = dataset_root / cfg.training_dataset
    test_split_root = dataset_root / cfg.test_dataset

    train_extractor = OutputSplitExtractor(
        split_root=train_split_root,
        dataset_root=dataset_root,
    )
    test_extractor = OutputSplitExtractor(
        split_root=test_split_root,
        dataset_root=dataset_root,
    )

    train_full_records = train_extractor.extract()
    test_records = test_extractor.extract()

    class_names = sorted(list({r["label_name"] for r in (train_full_records + test_records)}))
    label_to_idx = {name: i for i, name in enumerate(class_names)}

    train_records, val_records = train_test_split(
        train_full_records,
        test_size=cfg.val_ratio,
        random_state=SEED,
        stratify=[r["label_name"] for r in train_full_records],
    )

    train_ds = FaceShapeHybridDataset(train_records, label_to_idx, image_size=cfg.image_size)
    val_ds = FaceShapeHybridDataset(val_records, label_to_idx, image_size=cfg.image_size)
    test_ds = FaceShapeHybridDataset(test_records, label_to_idx, image_size=cfg.image_size)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

    return train_loader, val_loader, test_loader, label_to_idx


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total = 0

    for batch in loader:
        images = batch["image"].to(DEVICE)
        landmarks = batch["landmarks"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        logits = model(images, landmarks)
        loss = criterion(logits, labels)

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / max(total, 1)
    avg_acc = total_correct / max(total, 1)
    return avg_loss, avg_acc


In [ ]:
train_loader, val_loader, test_loader, label_to_idx = build_loaders(CFG)

sample_batch = next(iter(train_loader))
landmark_dim = sample_batch["landmarks"].shape[1]

model = HybridFaceShapeNet(
    num_classes=len(label_to_idx),
    landmark_dim=landmark_dim,
    image_embedding_dim=128,
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr)

print("classes:", label_to_idx)
print("landmark_dim:", landmark_dim)
print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))


In [ ]:
best_val = 0.0
for epoch in range(1, CFG.epochs + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer=optimizer)
    va_loss, va_acc = run_epoch(model, val_loader, criterion, optimizer=None)

    if va_acc > best_val:
        best_val = va_acc

    print(
        f"Epoch {epoch:02d}/{CFG.epochs} | "
        f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
        f"val_loss={va_loss:.4f} val_acc={va_acc:.4f}"
    )

te_loss, te_acc = run_epoch(model, test_loader, criterion, optimizer=None)
print("Best val_acc:", round(best_val, 4))
print(f"Test loss={te_loss:.4f} | Test acc={te_acc:.4f}")


In [ ]:
checkpoint_dir = Path("training_notebook") / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "approach2_mobilefacenet_hybrid.pt"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "label_to_idx": label_to_idx,
    "landmark_dim": landmark_dim,
    "image_embedding_dim": 128,
    "cfg": {
        "image_size": CFG.image_size,
        "batch_size": CFG.batch_size,
        "lr": CFG.lr,
        "epochs": CFG.epochs,
        "val_ratio": CFG.val_ratio,
        "dataset_root": CFG.dataset_root,
        "training_dataset": CFG.training_dataset,
        "test_dataset": CFG.test_dataset,
    },
}

torch.save(checkpoint, checkpoint_path)
print(f"Model saved to: {checkpoint_path.resolve()}")


In [ ]:
loaded_ckpt = torch.load(checkpoint_path, map_location=DEVICE)

loaded_model = HybridFaceShapeNet(
    num_classes=len(loaded_ckpt["label_to_idx"]),
    landmark_dim=loaded_ckpt["landmark_dim"],
    image_embedding_dim=loaded_ckpt.get("image_embedding_dim", 128),
).to(DEVICE)
loaded_model.load_state_dict(loaded_ckpt["model_state_dict"])

loaded_test_loss, loaded_test_acc = run_epoch(loaded_model, test_loader, criterion, optimizer=None)
print(f"Loaded-model test loss={loaded_test_loss:.4f} | test acc={loaded_test_acc:.4f}")
